In [1]:
import sys
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

spark = (SparkSession.builder
         .appName("check-python")
         .master("local[*]")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 16:07:06 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/03/03 16:07:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 16:07:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 16:07:07 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/03 16:07:07 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


---
Если вы работаете в качестве Data Scientist или Data Analyst, Вам часто приходится анализировать большие наборы данных/файлы с миллиардами или триллионами записей. 

Обработка таких больших наборов данных занимает некоторое время, поэтому на этапе анализа рекомендуется использовать случайную выборку из больших файлов.

**Использование PySpark SQL sample()**

Получение выборки PySpark (pyspark.sql.DataFrame.sample()) - это механизм получения случайных выборочных записей из набора данных. Это полезно, когда у вас есть большой набор данных и вы хотите проанализировать/протестировать подмножество данных, например, 10% от исходного файла.

In [19]:
df=spark.range(100)
print('\n',df.sample(fraction=0.06, seed=42).collect(),'\n')
df.show(5)


 [Row(id=44), Row(id=50), Row(id=58), Row(id=65)] 

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+
only showing top 5 rows


Мой DataFrame содержит 100 записей, и я хотел получить 6% выборочных записей, что равняется 4, но функция sample() вернула 5 записей. 

Это показывает, что функция sample не возвращает точно заданную долю (а лишь приблизительную).

In [10]:
print(df.sample(withReplacement=True,fraction=0.03,seed=42).collect()) #with Duplicates

[Row(id=14), Row(id=30), Row(id=81), Row(id=96)]


---
---
**Стратифицированная выборка в PySpark**

Функция sampleBy в PySpark используется для выборки случайных подмножеств данных из DataFrame на основе значений в определённом столбце. Эта функция полезна, когда необходимо выбрать определённые пропорции данных из различных категорий в столбце.

Основные моменты:

* Условная выборка: sampleBy позволяет выбирать данные на основе значений в одном из столбцов, при этом можно задать разные вероятности для каждого значения.

* Контроль над выборкой: Вы можете задать вероятность для каждой категории, чтобы контролировать размер подмножества, которое будет выбрано для каждой категории.

* Детерминированная выборка: Функция поддерживает использование параметра seed, который позволяет делать выборку детерминированной (воспроизводимой), если необходимо.

In [ ]:
df2 = df.select((df.id % 3).alias("key"))

print(df2.sampleBy("key", {0: 0.1, 1: 0.2},0).collect())

[Row(key=0), Row(key=0), Row(key=1), Row(key=1), Row(key=0), Row(key=1), Row(key=0), Row(key=1), Row(key=0), Row(key=0), Row(key=1), Row(key=1), Row(key=0)]


In [20]:
spark.stop()